In [11]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']


def generate_transaction():
    is_suspicious = random.random() < 0.05
    
    if is_suspicious:
        amount = round(random.uniform(3001.0, 5000.0), 2)
        category = 'elektronika'
        hour = random.randint(0, 5)
    else:
        amount = round(random.uniform(5.0, 2500.0), 2)
        category = random.choice(kategorie)
        hour = random.randint(6, 23)

    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': amount,
        'store': random.choice(sklepy),
        'category': category,
        'hour': hour,
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | Hour: {tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [8]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję tematu 'transactions' w poszukiwaniu dużych kwot...")

for message in consumer:
   
    tx = message.value
    
    if tx['amount'] > 1000:
        print(f"🚨 ALERT! Znaleziono dużą transakcję: ID={tx['tx_id']}, Kwota={tx['amount']:.2f} PLN, Sklep={tx['store']}")

Writing consumer_filter.py


In [10]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Uruchamiam proces wzbogacania danych (Enrichment)...")

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 3000:
        tx['risk_level'] = "HIGH"
    elif tx['amount'] > 1000:
        tx['risk_level'] = "MEDIUM"
    else:
        tx['risk_level'] = "LOW"
        
    print(f"[{tx['risk_level']:^6}] TX_ID: {tx['tx_id']} | Kwota: {tx['amount']:>7.2f} PLN | Sklep: {tx['store']}")

Writing consumer_enrich.py


In [12]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

print("Rozpoczynam zliczanie transakcji per sklep...")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1
    

    if msg_count % 10 == 0:
        print(f"\n--- PODSUMOWANIE (Przetworzono wiadomości: {msg_count}) ---")
        print(f"{'Sklep':<12} | {'Liczba TX':<10} | {'Suma PLN':<12}")
        print("-" * 40)
        
        # Iterujemy po wszystkich sklepach i wyświetlamy ich statystyki
        for s, count in store_counts.items():
            print(f"{s:<12} | {count:<10} | {total_amount[s]:>10.2f}")

Writing consumer_count.py


In [13]:
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []
    
    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1')
        

    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2')
        
    if 'hour' in tx:
        tx_hour = tx['hour']
    else:
        dt = datetime.fromisoformat(tx['timestamp'])
        tx_hour = dt.hour
        
    if tx_hour < 6:
        score += 2
        rules.append('R3')
        
    return score, rules

#Test
test_tx = {
    'tx_id': 'TX999', 
    'amount': 4500.0, 
    'category': 'elektronika',
    'timestamp': '2026-04-01T03:15:00'
}

wynik_score, trafione_reguly = score_transaction(test_tx)
czy_podejrzana = "TAK" if wynik_score >= 3 else "NIE"

print(f"Score: {wynik_score}")
print(f"Złamane reguły: {trafione_reguly}")
print(f"Podejrzana: {czy_podejrzana}")

Score: 7
Złamane reguły: ['R1', 'R2', 'R3']
Podejrzana: TAK


In [15]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []
    

    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1 (High Amount)')
        

    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2 (Expensive Tech)')

    tx_hour = tx.get('hour')
    if tx_hour is None and 'timestamp' in tx:
        tx_hour = datetime.fromisoformat(tx['timestamp']).hour
        
    if tx_hour is not None and tx_hour < 6:
        score += 2
        rules.append('R3 (Night Hours)')
        
    return score, rules

print("Silnik scoringowy uruchomiony. Analizuję transakcje...")

for message in consumer:
    tx = message.value
    score, triggered_rules = score_transaction(tx)
    

    if score >= 3:

        tx['fraud_score'] = score
        tx['triggered_rules'] = triggered_rules
        
     
        alert_producer.send('alerts', value=tx)
        
        print(f"🚨 ALERT: Wykryto podejrzaną transakcję!")
        print(f"   ID: {tx['tx_id']} | Punkty: {score} | Reguły: {triggered_rules}")
        print(f"   Szczegóły: {tx['amount']} PLN | {tx['category']} | Godzina: {tx.get('hour', 'Nieznana')}")
        print("-" * 30)


alert_producer.flush()

Overwriting scoring_consumer.py
